# Issue #47 — Method Discovery After the Negative Pilot

Preserve PR #48 and investigate a different mechanism before choosing another preprocessing candidate.

**Inputs:** Hash-verified committed PR #48 evidence and this study's saved diagnosis/resolution evidence. Explicit acquisition additionally needs original annotation metadata, saved matching caches and verified Skip JPEG95 exports.

**Outputs:** Read-only evidence displays by default; opted-in diagnosis and a single fixed 640/960 native diagnostic under docs/issue47-discovery and runs/issue47-discovery.

**Prerequisites:** Committed discovery plan and saved evidence; study dependencies/pinned weights only for explicitly enabled acquisition. No upstream notebooks execute. No new exploration, confirmation, development or final pixels are read.

<a id="section-1"></a>

## 1. Configuration

Read docs/ISSUE47_DISCOVERY_PLAN.md before opting in. Existing/incomplete acquisition attempts block re-execution. Default execution only reads committed artifacts, without native inference, GT loading or local runs.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from vehicle_detection.common import require_paths
from vehicle_detection.method_discovery import build_diagnosis, read_report, resolution

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
REPORT = ROOT / "docs/issue47-discovery"
require_paths(plan=ROOT / "docs/ISSUE47_DISCOVERY_PLAN.md")
ALLOW_DIAGNOSIS = False
ALLOW_RESOLUTION = False

<a id="section-2"></a>

## 2. Saved Failure Diagnosis

Class TP/FP/FN uses native confidence .25 / IoU .50 matches and original GT counts. Size distributions describe all GT, not individual missed boxes. Cohorts retain all GT in each image; their AP is not AP-small. Metadata and image quality describe associations, not causes.

In [ ]:
diagnosis = build_diagnosis(ROOT) if ALLOW_DIAGNOSIS else read_report(ROOT)
print(diagnosis["population"])
display(pd.DataFrame(diagnosis["size_support"]).T)
for group in (
    "all",
    "dark",
    "bright",
    "low_contrast",
    "higher_noise",
    "small_two_wheeler",
    "route_skip",
):
    print(group)
    display(pd.DataFrame(diagnosis["cohorts"]["skip"][group]["counts"]).T)
print(diagnosis["limit"])
print("Full per-class/cohort AP and original scene attributes remain in diagnosis.json.")

<a id="section-3"></a>

## 3. Ranked Mechanisms and Research Decision

The table records the premeasurement literature review and distinguishing controls for the completed 640/960 diagnostic. The current next priority is resolution-aware fine-tuning, described in section 5; task-aligned photometric training remains secondary. The committed measurement plan is unchanged. Spatial correction and one possible SCI comparator are deferred; no new enhancement framework/checkpoint is installed.

In [ ]:
display(
    pd.DataFrame(
        [
            {
                "Rank": 1,
                "Direction": "Native 640 versus 960 diagnostic",
                "Mechanism": "Retain more object detail",
                "Evidence": "Poor bright two-wheeler coverage; some small targets",
                "Control": "Same JPEG95/GT/model, only imgsz changes",
                "Budget": "2 native evaluations / 2,054 image passes",
                "Falsification": "Primary, two-wheeler, uncertainty, regression or cost gates fail",
            },
            {
                "Rank": 2,
                "Direction": "Task-aligned photometric training",
                "Mechanism": "Learn exposure robustness rather than only brighten inputs",
                "Evidence": "Enhancement did not rescue mostly skipped two-wheelers",
                "Control": "Future paired native short training; augmentation only",
                "Budget": "Not authorized in this round",
                "Falsification": "No supported held-out gain or clean-data harm",
            },
            {
                "Rank": 3,
                "Direction": "Constrained luminance/local contrast",
                "Mechanism": "Expose retained shadow structure while protecting highlights",
                "Evidence": "Current global features do not localize correctable target shadows",
                "Control": "Future local versus matched global correction",
                "Budget": "Deferred; no filter grid",
                "Falsification": "No detection benefit or amplified FP/noise",
            },
            {
                "Rank": 4,
                "Direction": "One possible SCI comparator",
                "Mechanism": "Learn spatial illumination",
                "Evidence": "External face/segmentation results are not BDD vehicle evidence",
                "Control": "One prospectively fixed checkpoint versus Skip",
                "Budget": "Deferred; license/checkpoint/CPU cost unresolved",
                "Falsification": "No native detection gain or excessive CPU cost",
            },
        ]
    ).set_index("Rank")
)

<a id="section-4"></a>

## 4. Controlled Resolution Diagnostic

Both evaluations use the already-consumed population. This is not an offline preprocessing policy, a cross-version S20 comparison, or independent confirmation. Native per-image mean timings and synchronized batch mean/p95 are reported separately; batch p95 is never called image p95. The original 640 result is checked for reproduction.

In [ ]:
result = resolution(ROOT, allow_inference=ALLOW_RESOLUTION)
display(pd.DataFrame(result["native"]).T)
display(result["timings"])
display(result["decision"])
interval = result["paired_interval"]
display(
    {
        k: interval[k]
        for k in ("recovery", "recovery_ci_low", "recovery_ci_high", "valid_draws", "invalid_draws")
    }
)
if (REPORT / "discovery.png").exists():
    display(Image(filename=str(REPORT / "discovery.png")))

<a id="section-5"></a>

## 5. Research Summary

**Next priority: a separately declared short paired 640/960 fine-tuning diagnostic.** Independently initialize both arms from the same pinned YOLO26n weights with identical train IDs/GT, epochs, seed, freeze and augmentation settings. Evaluate each arm at both 640 and 960 on the same consumed exploration IDs. This 2×2 matrix must test whether training at 960 adds benefit over the 640-trained arm evaluated at 960, rather than just reproduce the existing inference-resolution gain.

Declare the source-group-safe subset, compute budget, rare-class uncertainty and supported cohort/clean-regression stop criteria before running it. Report VRAM, training time and inference latency; disclose any necessary batch-size difference as a confounder. Task-aligned photometric training remains a secondary candidate, with no augmentation change in the resolution test.

Read docs/ISSUE47_DISCOVERY_RESULTS.md for the decision and limitations. This PR preserves the measured results and frozen plan, adds no inference/training, and does not select a preprocessing policy or promote 960 to a v3 S20 result. Issue #47 remains open; full S20 training still requires a justified candidate and independent confirmation.

In [ ]:
print("Preprocessing policy selected:", result["policy_selected"])
print("Independent confirmation:", result["decision"]["independent_confirmation"])
print("PR #48 and completed v0/v1/v2 artifacts remain unchanged; issue #47 stays open.")